<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/00_database_foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.0 · Database foundations

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~25 min**

## Learning objectives
- explain what a **relational database** and a **DBMS** are, and why they beat a spreadsheet;
- name the parts of a table and the common **data types**;
- distinguish **primary**, **composite** and **foreign keys**, and referential integrity;
- read **one-to-many** and **many-to-many** relationships (and the junction table);
- read our study's full **schema**;
- describe what **SQL** is and the **anatomy of a SELECT**.

Before we write a single query, this page gives you the conceptual base: what a database is, how tables link, and how to read our schema. Biology already runs on databases; GenBank, UniProt, SILVA and ENA are all databases you query every day. Read it once, and keep the schema in section 8 on screen for the rest of the day.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [1]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.6/397.6 kB 12.6 MB/s eta 0:00:00


Connecting to 'sqlite:///../data/omics.db'

Connected to omics.db — you are ready.


## 1. Data is everywhere in biology
Amplicon sequencing, metagenomics and metabarcoding produce tables with thousands to millions of rows, linked to sample metadata, taxonomy, gene functions… Storing that in one big spreadsheet quickly breaks down. The reference resources of our field, GenBank, UniProt, SILVA, ENA and KEGG, are all databases, precisely because they must store huge, linked data and let anyone query it reliably.

## 2. Flat file vs. relational database
Imagine one giant spreadsheet with a row per (sample, taxon) observation. The sample's pH and the taxon's phylum would be repeated on every row. That is wasteful, and if you fix a pH you must fix it in hundreds of places, and will miss some.

A relational database instead splits the data into a few tables that are linked by keys. Each fact is stored once. Benefits:
- no repetition (store each sample's pH once, in `samples`);
- consistency and integrity (the database refuses impossible data);
- fast, expressive queries with SQL, even over millions of rows.

## 3. What is a DBMS?
A **Database Management System** is the software that stores the tables, enforces the rules, and runs your SQL, for example SQLite, PostgreSQL, MySQL, DuckDB. The SQL you learn here is portable across all of them. We use SQLite: an entire database in a single `.db` file, with nothing to install, which suits learning and small omics projects well.

## 4. The relational model: tables, rows, columns
Data lives in **tables** (formally *relations*):
- a row = one record (one sample, one taxon), also called a *tuple*;
- a column = one property (its *attribute* or *field*), with a fixed data type;
- one table describes one kind of thing (one table for samples, one for taxa…).

A slice of our `taxa` table:

| taxon_id | genus | family | phylum | domain |
|---|---|---|---|---|
| T001 | Pseudomonas | Pseudomonadaceae | Pseudomonadota | Bacteria |
| T014 | Methanosarcina | Methanosarcinaceae | Euryarchaeota | Archaea |

↑ columns (attributes) · each line is a row (record).

## 5. Data types
Every column holds one type of value. The ones you will meet:

| Type | Holds | Example in our data |
|---|---|---|
| `TEXT` | text / strings | `environment` = 'Soil', `genus` = 'Bacillus' |
| `INTEGER` | whole numbers | `count` = 128 |
| `REAL` | decimal numbers | `ph` = 6.42, `temperature_c` = 14.7 |
| `NULL` | a **missing / unknown** value | a sample whose pH was not measured |

`NULL` is special: it means *“no value”*, and it needs `IS NULL`, never `= NULL`. You will use that in lesson 02.

## 6. Keys
**Primary key (PK):** a column (or set of columns) that **uniquely identifies each row**. It cannot repeat and cannot be NULL. In our database:
- `samples.sample_id`, `taxa.taxon_id`, `sites.site` are primary keys.

**Composite (compound) key:** a PK made of more than one column. `abundance` uses `(sample_id, taxon_id)` together, because one row means *one taxon in one sample*, and that pair is unique.

**Foreign key (FK):** a column that **points to another table's primary key**, linking the tables:
- `abundance.sample_id` → `samples.sample_id`
- `abundance.taxon_id` → `taxa.taxon_id`
- `samples.site` → `sites.site`

**Referential integrity** is the rule the DBMS enforces: you cannot record an abundance for a `sample_id` that does not exist in `samples`. Keys are also what a JOIN follows to reconnect the tables (lesson 05).

## 7. Relationships & cardinality
How many rows on each side link together?

- **One-to-many:** one `site` has many `samples`; each sample belongs to one site. (The `many` side, `samples`, carries the foreign key `site`.)
- **Many-to-many:** one `sample` contains many taxa, and one `taxon` occurs in many samples. A database cannot store many-to-many directly, so we use a junction (bridge) table: `abundance`. Each of its rows is one pairing, one taxon in one sample, plus the measured `count`.

> This junction pattern is the omics “feature table”: samples × features stored as one row per observed pairing. Recognising it makes much of SQL fall into place.

## 8. Our database schema (keep this on screen)
```
  sites                         taxa
  ┌───────────────┐             ┌──────────────┐
  │ site      (PK)│             │ taxon_id (PK)│
  │ country       │             │ genus        │
  │ latitude      │             │ family       │
  │ longitude     │             │ phylum       │
  └──────┬────────┘             │ domain       │
         │ 1                    └──────┬───────┘
         │                             │ 1
         │ many                        │
  ┌──────┴────────────┐         many   │
  │ samples           │ 1   ┌──────────┴─────────────┐
  │ sample_id     (PK)│─────│ abundance              │
  │ site          (FK)│ many│ sample_id (PK, FK)     │
  │ environment       │     │ taxon_id  (PK, FK)     │
  │ treatment         │     │ count                  │
  │ ph, temperature_c │     └────────────────────────┘
  │ depth_cm, date    │
  └───────────────────┘
```

| Table | Columns (type) | Keys |
|---|---|---|
| `sites` | site (TEXT), country (TEXT), latitude (REAL), longitude (REAL) | PK: site |
| `samples` | sample_id (TEXT), site (TEXT), environment (TEXT), treatment (TEXT), ph (REAL), temperature_c (REAL), depth_cm (REAL), collection_date (TEXT) | PK: sample_id · FK: site→sites |
| `taxa` | taxon_id (TEXT), genus (TEXT), family (TEXT), phylum (TEXT), domain (TEXT) | PK: taxon_id |
| `abundance` | sample_id (TEXT), taxon_id (TEXT), count (INTEGER) | PK: (sample_id, taxon_id) · FK: sample_id→samples, taxon_id→taxa |

## 9. What is SQL?
SQL (Structured Query Language) is a declarative language: you describe *what* you want, and the DBMS decides *how* to fetch it efficiently. It has three families:

| Family | Verbs | Purpose |
|---|---|---|
| **DQL** — queries | `SELECT` | ask questions (our whole focus, lessons 01–08) |
| **DML** — data | `INSERT`, `UPDATE`, `DELETE` | add/change/remove rows (lesson 09) |
| **DDL** — schema | `CREATE`, `ALTER`, `DROP` | define the tables themselves (lesson 09) |

SQL keywords are not case-sensitive (`SELECT` = `select`); we write them in capitals by convention. Statements end with a semicolon `;`.

## 10. Anatomy of a SELECT (your map for the whole day)
Almost every query you write fits this template, and the clauses always appear in this order:

```
SELECT   columns          -- WHICH columns to show
FROM     table            -- WHERE the data lives
WHERE    row condition    -- keep only some ROWS         (lesson 02)
GROUP BY columns          -- collapse rows into groups   (lesson 04)
HAVING   group condition  -- keep only some GROUPS       (lesson 04)
ORDER BY columns          -- sort the result             (lesson 03)
LIMIT    n;               -- keep only the first n rows  (lesson 03)
```

You only use the clauses you need (often just `SELECT … FROM … WHERE`). One subtlety worth knowing early: the database runs the clauses roughly in the order `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`, not top to bottom. That is why a name you create in `SELECT` can be used in `ORDER BY` but not in `WHERE`.

## 10b. Scalar vs. aggregate values (single number vs. summary)
Two questions look similar but return very different things:

- a scalar value is one value from one row, for example *"what is the pH of sample S001?"*;
- an aggregate value summarises many rows into one, for example *"what is the *average* pH over all samples?"*. It uses an aggregate function (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`).

The tell is the function: `ph` is a scalar column; `AVG(ph)` collapses the whole column to a single summary number. You will meet aggregates properly in lesson 04, but spotting the difference now saves confusion.

In [ ]:
%%sql
-- SCALAR: one value from one row (the pH of a specific sample)
SELECT sample_id, ph
FROM samples
WHERE sample_id = 'S001';

In [ ]:
%%sql
-- AGGREGATE: many rows collapsed into one summary number
-- (the mean pH across every sample; AVG ignores NULLs)
SELECT AVG(ph) AS mean_ph, MIN(ph) AS lowest, MAX(ph) AS highest
FROM samples;

## 10c. The result set
Every `SELECT` gives back one thing: a result set, itself a table (rows × named columns) held in memory, not stored in the database. That is why you can feed one query's result into another (a subquery, lesson 06) or save it as a `VIEW` (lesson 09): the output of SQL is the same shape as its input. A result set may have many rows, exactly one row (a scalar summary), or zero rows if nothing matched. Zero rows is a valid answer, not an error.

In [ ]:
%%sql
-- This result set has exactly the columns we name, in the order we name them.
-- (Rename freely with AS; the stored table is untouched.)
SELECT genus AS microbe, phylum AS lineage
FROM taxa
LIMIT 3;

## 10d. What a JOIN does, conceptually
Our data is split across tables, so most questions need columns from more than one. A JOIN matches rows from two tables on a shared key and stitches them into one wider row. Picture two tiny tables:

```
samples (left)                 sites (right)
┌────────────┬─────────┐       ┌──────────┬─────────┐
│ sample_id  │ site    │       │ site     │ country │
├────────────┼─────────┤       ├──────────┼─────────┤
│ S001       │ LEI_SOIL│       │ LEI_SOIL │ Germany │
│ S002       │ OSL_LAKE│       │ OSL_LAKE │ Norway  │
└────────────┴─────────┘       └──────────┴─────────┘
```
Joining `ON samples.site = sites.site` finds, for each sample, the site row with the **same `site` value**, and glues them together:
```
sample_id │ site     │ country
──────────┼──────────┼─────────
S001      │ LEI_SOIL │ Germany
S002      │ OSL_LAKE │ Norway
```
That is all a JOIN is: follow the foreign-key arrow in the schema and line up matching rows. You will write real joins in lesson 05.

In [ ]:
%%sql
-- A real join: attach each sample's country by matching site -> sites.site
SELECT s.sample_id, s.site, si.country
FROM samples s
JOIN sites si ON s.site = si.site
LIMIT 5;

## 10e. NULL and three-valued logic (why you need `IS NULL`)
A `NULL` means *"unknown / not measured"*, not zero and not an empty string. Because the value is unknown, SQL uses three-valued logic: a comparison can be TRUE, FALSE, or UNKNOWN. Any comparison with NULL, even `NULL = NULL`, is UNKNOWN, and `WHERE` keeps only rows that are TRUE. So `WHERE ph = NULL` returns nothing, ever. To test for missing values you must use the dedicated operators `IS NULL` and `IS NOT NULL`. (Aggregates like `AVG` quietly skip NULLs; you will rely on this in lesson 04.)

In [ ]:
%%sql
-- Correct way to find missing measurements: IS NULL (never = NULL)
SELECT sample_id, environment, ph
FROM samples
WHERE ph IS NULL;

## 10f. Two words you will hear: index & transaction
- an index is a behind-the-scenes lookup structure that makes searching a column fast (like the index at the back of a book). The DBMS uses it automatically; you rarely create one by hand for small data.
- a transaction is a group of changes applied all-or-nothing: either every change commits together or none do, so the database is never left half-updated (for example, a pipeline loading a whole sample's rows at once).

You do not need these to write queries today, but they are why databases stay fast and consistent at scale.

## 10g. Bridge: SQL vs. a spreadsheet, pandas, and dplyr
If you already use spreadsheets, pandas (Python course) or dplyr (R course), you already think in these operations; SQL just names them. Same ideas, different syntax:

| Operation | Spreadsheet | SQL | pandas (Python) | dplyr (R) |
|---|---|---|---|---|
| pick columns | hide/select columns | `SELECT col1, col2` | `df[["col1","col2"]]` | `select(col1, col2)` |
| keep some rows | filter/autofilter | `WHERE cond` | `df[df.cond]` | `filter(cond)` |
| sort | sort A→Z | `ORDER BY col` | `df.sort_values("col")` | `arrange(col)` |
| summarise per group | pivot table | `GROUP BY … + AVG()` | `df.groupby(...).mean()` | `group_by() %>% summarise()` |
| combine tables | VLOOKUP | `JOIN … ON` | `pd.merge(a, b, on=...)` | `left_join(a, b)` |

The big difference: a spreadsheet mixes data, formulas and formatting in one grid and struggles past ~100k rows; a database keeps clean typed tables and answers the same questions over millions of rows. Learn the concept once and you can move between all four tools.

## 11. See the database for real
Run these to connect and inspect the schema live.

In [ ]:
%%sql
-- the tables in this database
SELECT name FROM sqlite_master
WHERE type = 'table'
ORDER BY name;

In [ ]:
%%sql
-- how big is each table?
SELECT 'sites'     AS table_name, COUNT(*) AS n_rows FROM sites
UNION ALL SELECT 'samples',   COUNT(*) FROM samples
UNION ALL SELECT 'taxa',      COUNT(*) FROM taxa
UNION ALL SELECT 'abundance', COUNT(*) FROM abundance;

In [ ]:
%%sql
-- the columns and types of one table (PRAGMA is SQLite's way to inspect a table)
PRAGMA table_info(samples);

In [ ]:
%%sql
-- your first real question: how many samples per environment?
SELECT environment, COUNT(*) AS n_samples
FROM samples
GROUP BY environment;

In [ ]:
%%sql
-- another quick question: how many distinct phyla and domains do we track?
-- (COUNT(DISTINCT ...) is an aggregate over one column)
SELECT COUNT(DISTINCT phylum) AS n_phyla,
       COUNT(DISTINCT domain) AS n_domains
FROM taxa;

> **Instructor note.** Spend real time on sections 6–8 (keys and the schema). Every JOIN later is just following the FK arrows in the diagram; if students internalise that here, the rest of the day goes smoothly. Keep the schema visible throughout.

### Recap
- A relational database stores data in linked tables (rows × columns, typed).
- Types: `TEXT`, `INTEGER`, `REAL`, and `NULL` for missing values.
- Primary key = unique row id (may be composite, like `abundance`); foreign key = a pointer to another table's PK; the DBMS enforces integrity.
- One-to-many (site→samples) and many-to-many (samples↔taxa via the `abundance` junction table) are the relationships in our data.
- SQL is declarative; a `SELECT` follows a fixed clause order.
- Next: 01 · Meet the database & first SELECT.